# Arabic Sentiment Analysis

A reproducible study of Arabic sentiment classification on the ArSarcasm-v2 dataset.

## Notebook roadmap

1. Setup and configuration
2. Dataset loading and validation
3. Exploratory data analysis and splitting
4. Arabic text preprocessing
5. Feature representations
6. Model development and tuning
7. Preprocessing ablation
8. Data augmentation
9. Class-imbalance handling
10. Comparative evaluation and error analysis

## 1. Setup and configuration

This section loads the shared experiment configuration, fixes sources of randomness, and selects the available compute device.

In [1]:
from __future__ import annotations

import hashlib
import platform
import random
import shutil
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import yaml
from IPython.display import display

In [2]:
def find_project_root(start: Path) -> Path:
    """Find the repository root from a notebook or project-root kernel."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"

with CONFIG_PATH.open(encoding="utf-8") as config_file:
    CONFIG = yaml.safe_load(config_file)

PATHS = {
    name: PROJECT_ROOT / relative_path
    for name, relative_path in CONFIG["paths"].items()
}

CONFIG_PATH.relative_to(PROJECT_ROOT)

PosixPath('configs/default.yaml')

In [3]:
SEED = int(CONFIG["project"]["seed"])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if CONFIG["runtime"]["deterministic"]:
    torch.use_deterministic_algorithms(True, warn_only=True)

device_preference = CONFIG["runtime"]["device"]
if device_preference == "auto":
    device_name = "mps" if torch.backends.mps.is_available() else "cpu"
else:
    device_name = device_preference

DEVICE = torch.device(device_name)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(context="notebook", style="whitegrid")

In [4]:
pd.DataFrame(
    {
        "value": [
            CONFIG["project"]["name"],
            platform.python_version(),
            torch.__version__,
            str(DEVICE),
            SEED,
        ]
    },
    index=["project", "python", "pytorch", "device", "seed"],
)

,value
project,arabic-sentiment-analysis
python,3.12.4
pytorch,2.14.0
device,mps
seed,42


## 2. Dataset loading and validation

ArSarcasm-v2 contains Arabic tweets annotated for sentiment, sarcasm, and dialect. The dataset authors provide 12,548 training records and 3,000 testing records. Both files are fully labeled.

Sources:

- [Official ArSarcasm-v2 repository](https://github.com/iabufarha/ArSarcasm-v2)
- [WANLP 2021 task overview](https://aclanthology.org/2021.wanlp-1.36/)
- [Dataset license](https://github.com/iabufarha/ArSarcasm-v2/blob/main/LICENSE)

The source revision and SHA-256 digests are pinned below. Raw CSV files are downloaded into the Git-ignored `data/raw/` directory and are never modified in place.

In [5]:
DATASET_REVISION = "eb85513e133cfc67e30aca4968fadbe2594fcb42"
OFFICIAL_RAW_BASE = (
    "https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/"
    f"{DATASET_REVISION}/ArSarcasm-v2"
)

DATASET_MANIFEST = {
    "training": {
        "filename": "training_data.csv",
        "url": f"{OFFICIAL_RAW_BASE}/training_data.csv",
        "sha256": ("1da727eb763459f436c4e61f52c6a0bc63d5300af8fcc99b5675ae4616dae04a"),
        "rows": 12_548,
    },
    "testing": {
        "filename": "testing_data.csv",
        "url": f"{OFFICIAL_RAW_BASE}/testing_data.csv",
        "sha256": ("0f0a77ba8a4a0a9370846a42a93a63244092f1fb30a909595b70e17e299bcf6c"),
        "rows": 3_000,
    },
}

DATASET_MANIFEST

{'training': {'filename': 'training_data.csv',
  'url': 'https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/eb85513e133cfc67e30aca4968fadbe2594fcb42/ArSarcasm-v2/training_data.csv',
  'sha256': '1da727eb763459f436c4e61f52c6a0bc63d5300af8fcc99b5675ae4616dae04a',
  'rows': 12548},
 'testing': {'filename': 'testing_data.csv',
  'url': 'https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/eb85513e133cfc67e30aca4968fadbe2594fcb42/ArSarcasm-v2/testing_data.csv',
  'sha256': '0f0a77ba8a4a0a9370846a42a93a63244092f1fb30a909595b70e17e299bcf6c',
  'rows': 3000}}

In [6]:
def file_sha256(path: Path) -> str:
    """Calculate a file's SHA-256 digest without loading it into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_verified_file(specification: dict[str, object]) -> Path:
    """Download one immutable dataset file and verify its checksum."""
    raw_directory = PATHS["raw_data"]
    raw_directory.mkdir(parents=True, exist_ok=True)

    destination = raw_directory / str(specification["filename"])
    expected_digest = str(specification["sha256"])

    if destination.exists():
        observed_digest = file_sha256(destination)
        if observed_digest != expected_digest:
            raise ValueError(f"Checksum mismatch for existing file: {destination.name}")
        return destination

    temporary_path = destination.with_suffix(".download")
    temporary_path.unlink(missing_ok=True)

    try:
        with (
            urlopen(str(specification["url"]), timeout=60) as response,
            temporary_path.open("wb") as target,
        ):
            shutil.copyfileobj(response, target)

        observed_digest = file_sha256(temporary_path)
        if observed_digest != expected_digest:
            raise ValueError(f"Checksum mismatch after downloading {destination.name}")
        temporary_path.replace(destination)
    finally:
        temporary_path.unlink(missing_ok=True)

    return destination


DATASET_PATHS = {
    split_name: download_verified_file(specification)
    for split_name, specification in DATASET_MANIFEST.items()
}

{name: path.relative_to(PROJECT_ROOT) for name, path in DATASET_PATHS.items()}

{'training': PosixPath('data/raw/training_data.csv'),
 'testing': PosixPath('data/raw/testing_data.csv')}

In [7]:
EXPECTED_DIALECTS = {"egypt", "gulf", "levant", "magreb", "msa"}


def validate_dataset(
    frame: pd.DataFrame,
    split_name: str,
    expected_rows: int,
) -> None:
    """Reject incomplete or structurally unexpected source data."""
    expected_columns = list(CONFIG["data"]["expected_columns"])
    expected_sentiments = set(CONFIG["data"]["classes"])

    problems = []
    if len(frame) != expected_rows:
        problems.append(f"expected {expected_rows} rows, found {len(frame)}")
    if list(frame.columns) != expected_columns:
        problems.append("unexpected columns or column order")
    if frame.isna().any().any():
        problems.append("missing values found")
    if set(frame["sentiment"].unique()) != expected_sentiments:
        problems.append("unexpected sentiment labels")
    if set(frame["dialect"].unique()) != EXPECTED_DIALECTS:
        problems.append("unexpected dialect labels")
    if not pd.api.types.is_bool_dtype(frame["sarcasm"]):
        problems.append("sarcasm must be boolean")

    normalized_text = (
        frame["tweet"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    )
    if normalized_text.eq("").any():
        problems.append("blank tweet text found")

    if problems:
        raise ValueError(f"{split_name}: {'; '.join(problems)}")


DATASET_FRAMES = {}
for split_name, specification in DATASET_MANIFEST.items():
    frame = pd.read_csv(DATASET_PATHS[split_name])
    validate_dataset(frame, split_name, int(specification["rows"]))
    DATASET_FRAMES[split_name] = frame

raw_data = pd.concat(
    [
        frame.assign(source_split=split_name, source_row=np.arange(len(frame)))
        for split_name, frame in DATASET_FRAMES.items()
    ],
    ignore_index=True,
)

raw_data.shape

(15548, 6)

In [8]:
source_summary = pd.DataFrame(
    [
        {
            "source_split": split_name,
            "rows": len(DATASET_FRAMES[split_name]),
            "columns": len(DATASET_FRAMES[split_name].columns),
            "missing_values": int(DATASET_FRAMES[split_name].isna().sum().sum()),
            "sha256": file_sha256(DATASET_PATHS[split_name]),
        }
        for split_name in DATASET_MANIFEST
    ]
)

sentiment_distribution = pd.crosstab(
    raw_data["sentiment"],
    raw_data["source_split"],
    margins=True,
).rename_axis(index="sentiment", columns="source")

display(source_summary)
display(sentiment_distribution)

,source_split,rows,columns,missing_values,sha256
0,training,12548,4,0,1da727eb763459f436c4e61f52c6a0bc63d5300af8fcc9...
1,testing,3000,4,0,0f0a77ba8a4a0a9370846a42a93a63244092f1fb30a909...


source,testing,training,All
sentiment,,,
NEG,1677,4621,6298
NEU,748,5747,6495
POS,575,2180,2755
All,3000,12548,15548


In [9]:
audit_text = (
    raw_data["tweet"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
)
duplicate_mask = audit_text.duplicated(keep=False)

duplicate_groups = (
    raw_data.assign(_audit_text=audit_text)
    .groupby("_audit_text", sort=False)
    .agg(
        rows=("tweet", "size"),
        source_splits=("source_split", "nunique"),
        sentiment_labels=("sentiment", "nunique"),
        sarcasm_labels=("sarcasm", "nunique"),
        dialect_labels=("dialect", "nunique"),
    )
)
duplicate_groups = duplicate_groups[duplicate_groups["rows"] > 1]

duplicate_audit = pd.Series(
    {
        "combined_rows": len(raw_data),
        "exact_duplicate_rows": int(
            raw_data.duplicated(
                subset=["tweet", "sarcasm", "sentiment", "dialect"]
            ).sum()
        ),
        "rows_in_duplicate_text_groups": int(duplicate_mask.sum()),
        "duplicate_text_groups": len(duplicate_groups),
        "groups_crossing_source_splits": int(
            duplicate_groups["source_splits"].gt(1).sum()
        ),
        "groups_with_sentiment_conflicts": int(
            duplicate_groups["sentiment_labels"].gt(1).sum()
        ),
        "groups_with_sarcasm_conflicts": int(
            duplicate_groups["sarcasm_labels"].gt(1).sum()
        ),
        "groups_with_dialect_conflicts": int(
            duplicate_groups["dialect_labels"].gt(1).sum()
        ),
    },
    name="count",
).to_frame()

duplicate_audit

,count
combined_rows,15548
exact_duplicate_rows,18
rows_in_duplicate_text_groups,132
duplicate_text_groups,65
groups_crossing_source_splits,2
groups_with_sentiment_conflicts,26
groups_with_sarcasm_conflicts,20
groups_with_dialect_conflicts,21


### Validation notes

The notebook deliberately reports duplicate counts without displaying tweet contents. Duplicate normalized texts will be treated as groups during the new 60/20/20 split so equivalent text cannot leak across partitions. Conflicting duplicate annotations will be resolved by an explicit rule in the splitting section rather than silently overwritten here.